## Light curve

    Features:
    - Find asteroid location using potutils centroid_2dg
    - Find reference star location using query via Gaia
    - Light curve plot alongside error propagation

Author: Roberto S.

In [1]:
# IMPORTS

import numpy as np

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
%matplotlib qt

from astropy.io import fits
from astropy import units as u
from astropy.coordinates import Angle, SkyCoord
from astropy.wcs import WCS, FITSFixedWarning
from astropy.time import Time

from astroquery.jplhorizons import Horizons
from astroquery.gaia import Gaia

from photutils.centroids import centroid_2dg
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry

import warnings
warnings.simplefilter('ignore', FITSFixedWarning)

from scipy.optimize import minimize_scalar

import os

# ONLY WORKS ON MY LAPTOP, COULDN'T GET THE PATH TO WORK FOR THE REMOTE REPO
direc = r"C:\Users\rober\OneDrive\Documents\_Docs\Year 4\TGP\TGP_Asteroids2_2025"

c:\Users\rober\OneDrive\Documents\_Docs\Year 4\TGP\TGP_Asteroids2_2025\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# FIND SNR

def snr(data, ap, an):

    phot = aperture_photometry(data, ap)
    flux_ap = phot['aperture_sum'][0]

    annulus_masks = an.to_mask(method='center')
    annulus_data = annulus_masks.multiply(data)
    sky_pixels = annulus_data[annulus_masks.data > 0]
    sky_mean = np.mean(sky_pixels)
    sky_std  = np.std(sky_pixels)

    n_pix = ap.area
    flux = flux_ap - sky_mean * n_pix

    gain = 1.39
    flux_e = flux * gain
    sigma_sky_e = sky_std * gain

    sigma_flux = np.sqrt(flux_e + n_pix * sigma_sky_e**2)

    return flux/sigma_flux

# ASTEROID

In [3]:
# APPROX LOCS BASED ON EPHEMERIS

codes1 = [i for i in range(167820, 167861)if i not in
         [167825, 167830, 167831, 167832, 167845, 167847, 167848, 167849, 167850, 167851, 167852]]
codes2 = [167137, 167140, 167148, 167150, 167152, 167154, 167156,
          167158, 167159, 167161, 167163, 167165, 167167]
codes = [str(c) for c in codes1]

data = []
locs_app = []
hdrs = []

fig,axs = plt.subplots(5,6, figsize=(16,10))
for a,c in zip(axs.flatten(), codes):
    with fits.open(direc + r'\\Reduced_data_day3\\' + str(c) + '.fits') as file:

        hdr = file[0].header.copy()

        obj = Horizons(id="Belgica", location="Z63", epochs=Time(hdr["DATE-OBS"], format="isot", scale="utc").jd)
        eph = obj.ephemerides()
        ra = eph["RA"][0]
        dec = eph["DEC"][0]

        # correct for cropping the raw data
        hdr["CRPIX1"] -= 21
        hdr["CRPIX2"] -= 21
        wcs = WCS(hdr)

        px, py = wcs.world_to_pixel_values(ra, dec)
        circle = Circle((px, py), radius=10, edgecolor="red", facecolor="none", linewidth=2)

        image = file[0].data

        vmin, vmax = np.percentile(image, [5, 99])
        a.imshow(image, cmap='gray', vmin=vmin, vmax=vmax)
        a.add_patch(circle)

        data.append(image)
        locs_app.append((px, py))
        hdrs.append(hdr)

        # print(f"{c}:")
        # print(f"asteroid coords:\t{px:.1f}\t{py:.1f}")
        # print(f"image coords:\t\t{hdr['RA']:.1f}\t{hdr['DEC']:.1f}")
        # print()

plt.show()

for h,c in zip(hdrs, codes):
    if h["FILTER"] != "R": print(c)

In [4]:
# EXACT LOCS BASED ON GAUSSIAN FITTING

r = 10

fig,axs = plt.subplots(5,6, figsize=(16,10))

locs = []

for d,l,a,h in zip(data, locs_app, axs.flatten(), hdrs):

    x0, y0 = int(np.round(l[0])), int(np.round(l[1]))

    cutout = d[y0-r:y0+r+1, x0-r:x0+r+1]

    x_c, y_c = centroid_2dg(cutout)
  
    x_refined = x0 - r + x_c
    y_refined = y0 - r + y_c

    vn, vx = np.percentile(d, [5,99])
    a.imshow(d, cmap="gray", vmin=vn, vmax=vx)
    a.add_patch(Circle((x_refined, y_refined), radius=10, edgecolor="red", facecolor="none", linewidth=2))
    a.add_patch(Circle((x0, y0), radius=10, edgecolor="b", facecolor="none", linewidth=2))

    a.set_xlim(x_refined-20, x_refined+20)
    a.set_ylim(y_refined-20, y_refined+20)

    locs.append((x_refined, y_refined))

plt.show()

In [5]:
# CURVES OF GROWTH

ap_sizes = [i*.5 for i in range (5,18)]

f,axs = plt.subplots(5,6, figsize=(16,10))

for a,d,l in zip(axs.flatten(),data,locs):

    counts = []

    for s in ap_sizes:
        ap = CircularAperture((l[0], l[1]), r=s)
        asteroid = aperture_photometry(d, ap)
        counts.append(asteroid["aperture_sum"].value)

    a.plot(ap_sizes, counts, c="k")
    a.plot([5,5], [min(counts), max(counts)], c="r")

plt.show()

In [6]:
# SHOW APERTURE, ANNULUS

r_ap = 3
an1 = 2 * r_ap
an2 = 3 * r_ap

f,axs = plt.subplots(5,6, figsize=(16,10))

for a,d,l in zip(axs.flatten(),data,locs):

    vn, vx = np.percentile(d, [5,99])
    a.imshow(d, cmap="gray", vmin=vn, vmax=vx)

    a.add_patch(Circle((l[0],l[1]), radius=r_ap, edgecolor="r", facecolor="none", linewidth=1))

    a.add_patch(Circle((l[0],l[1]), radius=an1, edgecolor="b", facecolor="none", linewidth=1))
    a.add_patch(Circle((l[0],l[1]), radius=an2, edgecolor="b", facecolor="none", linewidth=1))

    a.set_xlim(l[0]-20, l[0]+20)
    a.set_ylim(l[1]-20, l[1]+20)

plt.show()

In [7]:
# ASTEROID COUNTS

astd_mags = []
astd_errs = []
times = []

t0 = Time(hdrs[0]["DATE-OBS"], format="isot", scale="utc")

for d,l,h in zip(data, locs, hdrs):

    ap = CircularAperture((l[0], l[1]), r=r_ap)
    an = CircularAnnulus((l[0], l[1]), r_in=an1, r_out=an2)

    sky = aperture_photometry(d,an)["aperture_sum"].value/an.area

    source = aperture_photometry(d-sky,ap)["aperture_sum"].value

    astd_mags.append(-2.5 * np.log10(source))
    astd_errs.append((2.5/np.log(10)) * (1/snr(d, ap, an)))
    times.append((Time(h["DATE-OBS"], format="isot", scale="utc")-t0).to("h").value)

# STAR

In [8]:
# FIND COMPARISON STAR IN 1ST IMAGE

obj = Horizons(id="Belgica", location="Z63", epochs=Time(hdrs[0]["DATE-OBS"], format="isot", scale="utc").jd)
eph = obj.ephemerides()
ra = eph["RA"][0]
dec = eph["DEC"][0]

centre = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame='icrs')

radius = 10 * u.arcmin

job = Gaia.cone_search_async(centre, radius=radius)
gaia_table = job.get_results()

stars = gaia_table[(gaia_table['phot_g_mean_mag'] > 10) &
                   (gaia_table['phot_g_mean_mag'] < 15)]
stars = stars[stars['ruwe'] < 1.4]

comp_star = stars[np.argmin(stars['phot_g_mean_mag'])]
ra_star, dec_star = comp_star["ra"], comp_star["dec"]
print(ra_star, dec_star)

INFO: Query finished. [astroquery.utils.tap.core]
346.4592596801406 -14.503958167625802


In [9]:
# PLOT STAR IN 1ST IMAGE

wcs = WCS(hdrs[0])
x_star_app, y_star_app = wcs.world_to_pixel_values(ra_star, dec_star)

f,a = plt.subplots(figsize=(16,10))
d = data[0]

vn, vx = np.percentile(d, [5,99])
a.imshow(d, cmap="gray", vmin=vn, vmax=vx)
a.add_patch(Circle((locs[0][0],locs[0][1]), radius=10, edgecolor="r", facecolor="none", linewidth=1))
a.add_patch(Circle((x_star_app,y_star_app), radius=10, edgecolor="orange", facecolor="none", linewidth=1))

plt.show()

In [10]:
# LOCATE EXACT STAR CENTRE IN EACH IMAGE

r = 10

fig,axs = plt.subplots(5,6, figsize=(16,10))

star_locs = []

for d,h,a in zip(data,hdrs,axs.flatten()):

    wcs = WCS(h)
    x0, y0 = wcs.world_to_pixel_values(ra_star, dec_star)
    a.add_patch(Circle((x0, y0), radius=10, edgecolor="r", facecolor="none", linewidth=2))
    x0, y0 = int(np.round(x0)), int(np.round(y0))

    cutout = d[y0-r:y0+r+1, x0-r:x0+r+1]

    x_c, y_c = centroid_2dg(cutout)

    x_refined = x0 - r + x_c
    y_refined = y0 - r + y_c

    star_locs.append((x_refined, y_refined))

    vn,vx = np.percentile(d, [5,99])
    a.imshow(d, cmap="gray", vmin=vn, vmax=vx)
    a.add_patch(Circle((x_refined, y_refined), radius=10, edgecolor="g", facecolor="none", linewidth=2))

    a.set_xlim(x_refined-20, x_refined+20)
    a.set_ylim(y_refined-20, y_refined+20)

plt.show()

In [11]:
# IMAGE 1 APERTURE RADIUS

ap_sizes = [i for i in range (1,13)]
counts = []

f,a = plt.subplots(1,2, figsize=(16,8))

vn, vx = np.percentile(data[0], [5,99])
a[0].imshow(data[0], cmap="gray", vmin=vn, vmax=vx)

for s in ap_sizes:
    ap = CircularAperture((star_locs[0][0], star_locs[0][1]), r=s)
    star = aperture_photometry(data[0], ap)
    counts.append(star["aperture_sum"].value)
    a[0].add_patch(Circle((star_locs[0][0], star_locs[0][1]), radius=s, edgecolor="r", facecolor="none", linewidth=1))
    a[0].set_xlim(star_locs[0][0]-20, star_locs[0][0]+20)
    a[0].set_ylim(star_locs[0][1]-20, star_locs[0][1]+20)

a[1].plot(ap_sizes, counts, c="k")
a[1].set_xlabel("aperture size (pix)")
a[1].set_ylabel("counts")
plt.show()

In [12]:
# CURVES OF GROWTH

ap_sizes = [i*.5 for i in range (5,18)]

f,axs = plt.subplots(5,6, figsize=(16,10))

for a,d,l in zip(axs.flatten(),data,star_locs):

    counts = []

    for s in ap_sizes:
        ap = CircularAperture((l[0], l[1]), r=s)
        star = aperture_photometry(d, ap)
        counts.append(star["aperture_sum"].value)

    a.plot(ap_sizes, counts, c="k")
    a.plot([6,6], [min(counts), max(counts)], c="r")

plt.show()

In [13]:
# SHOW APERTURE, ANNULUS

r_ap = 4
an1 = 2 * r_ap
an2 = 3 * r_ap

f,axs = plt.subplots(5,6, figsize=(16,10))

for a,d,l in zip(axs.flatten(),data,star_locs):

    vn, vx = np.percentile(d, [5,99])
    a.imshow(d, cmap="gray", vmin=vn, vmax=vx)

    a.add_patch(Circle((l[0],l[1]), radius=r_ap, edgecolor="g", facecolor="none", linewidth=1))

    a.add_patch(Circle((l[0],l[1]), radius=an1, edgecolor="c", facecolor="none", linewidth=1))
    a.add_patch(Circle((l[0],l[1]), radius=an2, edgecolor="c", facecolor="none", linewidth=1))

    a.set_xlim(l[0]-20, l[0]+20)
    a.set_ylim(l[1]-20, l[1]+20)

plt.show()

In [14]:
# STAR COUNTS

star_mags = []
star_errs = []

for d,l,h in zip(data, star_locs, hdrs):

    ap = CircularAperture((l[0], l[1]), r=r_ap)
    an = CircularAnnulus((l[0], l[1]), r_in=an1, r_out=an2)

    sky = aperture_photometry(d,an)["aperture_sum"].value/an.area

    source = aperture_photometry(d-sky,ap)["aperture_sum"].value

    star_mags.append(-2.5 * np.log10(source))
    star_errs.append((2.5/np.log(10)) * (1/snr(d, ap, an)))

# LIGHT CURVE

In [15]:
# PLOT LIGHT CURVE

f,a = plt.subplots(figsize=(16,10))

astd_mags = np.array(astd_mags)
star_mags = np.array(star_mags)

a_err = np.array([float(e) for e in astd_errs])
s_err = np.array([float(e) for e in star_errs])

rel_mags = astd_mags - star_mags
err_mags = a_err + s_err

# a.plot(times, astd_mags-star_mags, marker=".", c="k", ms=10)
a.errorbar(times, rel_mags.flatten(), err_mags, c="k")
a.set_xlabel("Time (hrs)")
a.set_ylabel("Asteroid-Star magnitude")

plt.show()

# APERTURE SNR OPTIMISATION

In [33]:
# USE FIRST IMAGE FOR SNR OPTIMISATION

d_ind = 0

def ap_opt(r):

    an1 = 2*r
    an2 = 3*r
    ap = CircularAperture((locs[d_ind][0], locs[d_ind][1]), r=r)
    an = CircularAnnulus((locs[d_ind][0], locs[d_ind][1]), r_in=an1, r_out=an2)

    return -snr(data[d_ind], ap, an)

res = minimize_scalar(ap_opt, bounds=(3, 20))
print(f"SNR {-res.fun:.2f} at aperture {res.x:.2f} pix")

f,a = plt.subplots(1,2, figsize=(16,10))

vn,vx = np.percentile(data[d_ind], [5,99])
a[0].imshow(data[d_ind], cmap="gray", vmin=vn, vmax=vx)
a[0].add_patch(Circle((locs[0][0],locs[0][1]), radius=res.x, edgecolor="g", facecolor="none", linewidth=1))
a[0].add_patch(Circle((locs[0][0],locs[0][1]), radius=res.x*2, edgecolor="c", facecolor="none", linewidth=1))
a[0].add_patch(Circle((locs[0][0],locs[0][1]), radius=res.x*3, edgecolor="c", facecolor="none", linewidth=1))
a[0].set_xlim(locs[d_ind][d_ind]-20, locs[0][0]+20)
a[0].set_ylim(locs[d_ind][1]-20, locs[d_ind][1]+20)

aps = np.linspace(0.5, 20, 200)
a[1].plot(aps, [-ap_opt(ap) for ap in aps], c="k")
a[1].scatter(res.x, -res.fun, c="k")

plt.show()


SNR 166.59 at aperture 3.03 pix


In [35]:
# USE FIRST IMAGE FOR SNR OPTIMISATION

f,axs = plt.subplots(5,6, figsize=(16,10))

optims = []

for d_ind,a in zip(range(30), axs.flatten()):

    def ap_opt(r):

        an1 = 2*r
        an2 = 3*r
        ap = CircularAperture((locs[d_ind][0], locs[d_ind][1]), r=r)
        an = CircularAnnulus((locs[d_ind][0], locs[d_ind][1]), r_in=an1, r_out=an2)

        return -snr(data[d_ind], ap, an)

    aps = np.linspace(0.5, 20, 20)
    rough_peak = aps[np.argmax([-ap_opt(ap) for ap in aps])]

    res = minimize_scalar(ap_opt, bounds=(rough_peak-2, rough_peak+2))
    print(f"SNR {-res.fun:.2f} at aperture {res.x:.2f} pix")
    optims.append(res.x)

    aps = np.linspace(rough_peak-2, rough_peak+2, 200)
    a.plot(aps, [-ap_opt(ap) for ap in aps], c="k")
    a.scatter(res.x, -res.fun, c="k")
    a.set_title(f"{res.x:.2f}")

plt.show()

print(f"\nOptimum {np.mean(optims):.2f} pix")

# 3 PIX OPTIMISES ASTEROID SNR

SNR 167.13 at aperture 2.93 pix
SNR 172.39 at aperture 3.08 pix
SNR 184.22 at aperture 2.84 pix
SNR 181.85 at aperture 3.47 pix
SNR 188.88 at aperture 2.97 pix
SNR 191.57 at aperture 3.03 pix
SNR 185.89 at aperture 2.99 pix
SNR 171.21 at aperture 3.51 pix
SNR 195.75 at aperture 2.44 pix
SNR 201.11 at aperture 3.30 pix
SNR 207.38 at aperture 2.96 pix
SNR 200.54 at aperture 3.30 pix
SNR 201.42 at aperture 3.00 pix
SNR 207.24 at aperture 2.60 pix
SNR 205.09 at aperture 2.84 pix
SNR 182.31 at aperture 2.54 pix
SNR 196.50 at aperture 2.82 pix
SNR 201.45 at aperture 2.45 pix
SNR 203.02 at aperture 2.53 pix
SNR 197.63 at aperture 3.15 pix
SNR 202.70 at aperture 2.56 pix
SNR 211.15 at aperture 2.51 pix
SNR 210.67 at aperture 2.80 pix
SNR 207.52 at aperture 2.64 pix
SNR 204.99 at aperture 3.17 pix
SNR 203.05 at aperture 2.79 pix
SNR 202.32 at aperture 2.84 pix
SNR 202.59 at aperture 2.62 pix
SNR 186.33 at aperture 3.78 pix
SNR 195.10 at aperture 3.09 pix

Optimum 2.92 pix


In [ ]:
# USE FIRST IMAGE FOR SNR OPTIMISATION

f,axs = plt.subplots(5,6, figsize=(16,10))

optims = []

for d_ind,a in zip(range(30), axs.flatten()):

    def ap_opt(r):

        an1 = 2*r
        an2 = 3*r
        ap = CircularAperture((star_locs[d_ind][0], star_locs[d_ind][1]), r=r)
        an = CircularAnnulus((star_locs[d_ind][0], star_locs[d_ind][1]), r_in=an1, r_out=an2)

        return -snr(data[d_ind], ap, an)

    aps = np.linspace(0.5, 20, 50)
    new = aps[np.argmax([-ap_opt(ap) for ap in aps])]
    if new > 2: rough_peak = new
    res = minimize_scalar(ap_opt, bounds=(rough_peak-2 if rough_peak-2>0 else 0.1, rough_peak+2))
    print(f"{d_ind}: SNR {-res.fun:.2f} at aperture {res.x:.2f} pix, rough at {rough_peak:.2f} pix")
    optims.append(res.x)

    aps = np.linspace(rough_peak-2 if rough_peak-2>0 else 0.1, rough_peak+2, 200)
    a.plot(aps, [-ap_opt(ap) for ap in aps], c="k")
    a.scatter(res.x, -res.fun, c="k")
    a.set_title(f"{res.x:.2f}")

plt.show()

print(f"\nOptimum {np.mean(optims):.2f} pix")

# 4 PIX OPTIMISES STAR SNR

0: SNR 427.94 at aperture 2.96 pix, rough at 2.89 pix
1: SNR 455.61 at aperture 3.14 pix, rough at 3.29 pix
2: SNR 458.13 at aperture 3.59 pix, rough at 3.68 pix
3: SNR 454.67 at aperture 3.85 pix, rough at 3.68 pix
4: SNR 468.80 at aperture 3.96 pix, rough at 3.68 pix
5: SNR 462.55 at aperture 3.87 pix, rough at 3.68 pix
6: SNR 462.91 at aperture 3.76 pix, rough at 3.68 pix
7: SNR 467.78 at aperture 3.77 pix, rough at 3.68 pix
8: SNR 461.82 at aperture 3.71 pix, rough at 3.68 pix
9: SNR 474.41 at aperture 3.86 pix, rough at 4.08 pix
10: SNR 468.58 at aperture 3.80 pix, rough at 3.68 pix
11: SNR 482.60 at aperture 3.63 pix, rough at 3.68 pix
12: SNR 475.77 at aperture 3.80 pix, rough at 3.68 pix
13: SNR 473.55 at aperture 3.92 pix, rough at 4.08 pix
14: SNR 427.04 at aperture 4.90 pix, rough at 6.07 pix
15: SNR 474.25 at aperture 3.91 pix, rough at 3.68 pix
16: SNR 479.62 at aperture 3.59 pix, rough at 3.68 pix
17: SNR 480.41 at aperture 3.51 pix, rough at 3.68 pix
18: SNR 481.28 at ap

c:\Users\rober\OneDrive\Documents\_Docs\Year 4\TGP\TGP_Asteroids2_2025\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\rober\OneDrive\Documents\_Docs\Year 4\TGP\TGP_Asteroids2_2025\.venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\rober\OneDrive\Documents\_Docs\Year 4\TGP\TGP_Asteroids2_2025\.venv\Lib\site-packages\numpy\_core\_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\rober\OneDrive\Documents\_Docs\Year 4\TGP\TGP_Asteroids2_2025\.venv\Lib\site-packages\numpy\_core\_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\rober\OneDrive\Documents\_Docs\Year 4\TGP\TGP_Asteroids2_2025\.venv\Lib\site-packages\numpy\

25: SNR 481.97 at aperture 3.35 pix, rough at 3.29 pix
26: SNR 469.04 at aperture 3.38 pix, rough at 3.29 pix


c:\Users\rober\OneDrive\Documents\_Docs\Year 4\TGP\TGP_Asteroids2_2025\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\rober\OneDrive\Documents\_Docs\Year 4\TGP\TGP_Asteroids2_2025\.venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\rober\OneDrive\Documents\_Docs\Year 4\TGP\TGP_Asteroids2_2025\.venv\Lib\site-packages\numpy\_core\_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\rober\OneDrive\Documents\_Docs\Year 4\TGP\TGP_Asteroids2_2025\.venv\Lib\site-packages\numpy\_core\_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
c:\Users\rober\OneDrive\Documents\_Docs\Year 4\TGP\TGP_Asteroids2_2025\.venv\Lib\site-packages\numpy\

27: SNR 465.99 at aperture 3.65 pix, rough at 3.68 pix
28: SNR 460.98 at aperture 3.50 pix, rough at 4.48 pix
29: SNR 460.92 at aperture 3.58 pix, rough at 3.68 pix

Optimum 3.68 pix
